# 03 - Train, tracked in MLflow

A SageMaker XGBoost training job, wrapped in an MLflow run.

### Where the two systems meet

The built-in XGBoost container reads plain CSV from S3. It cannot read a DVC
remote, where data is content-addressed. So:

1. `dvc repro` / `dvc checkout` materialises `data/processed/*.csv` locally.
2. `churn.training` copies train + validation to a **run-scoped** S3 prefix.
3. The MLflow run records the hyperparameters, the job name, those S3 URIs,
   **and the DVC hashes from `dvc.lock`**.

Those S3 copies are disposable. `dvc.lock` plus the git commit are the durable
record of what the model saw — which is why both are MLflow tags.

In [1]:
import json
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
os.environ["CHURN_PROJECT_ROOT"] = str(PROJECT_ROOT)

# Put this kernel's venv bin/ first on PATH so shell commands (dvc, ...) resolve to the
# venv, however Jupyter was launched.
import sys
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"
for line in (PROJECT_ROOT / ".env").read_text().strip().splitlines():
    key, _, value = line.partition("=")
    os.environ[key] = value

from churn.config import Settings, load_params

params = load_params()
settings = Settings.load(params)
print("region:", settings.region, "| bucket:", settings.bucket)

region: us-east-1 | bucket: sagemaker-us-east-1-063676201336


## 1. Verify the data is materialised

Training against a stale working tree is the easiest way to produce a run whose
tags lie. Check before spending money on a job.

In [2]:
import subprocess

status = subprocess.run(["dvc", "status"], capture_output=True, text=True, cwd=PROJECT_ROOT)
print(status.stdout or status.stderr)

if "up to date" not in (status.stdout + status.stderr).lower():
    print("\n!! Working tree does not match dvc.lock. Run `dvc repro` (notebook 02) first.")

Data and pipelines are up to date.



In [3]:
from churn.dvc_meta import describe, lineage_tags

print("this run will be tagged:\n")
for key, value in sorted(lineage_tags().items()):
    print(f"  {key:30s} {value}")
print("\n" + describe())

this run will be tagged:

  dvc.data.interim.features.parquet d0605019cf55f50aaeee3c993fe9847a
  dvc.data.processed             5f68eff31363915f49ff8091544bd6e0.dir
  dvc.reports.split_summary.json bf9ed4c83e081dac839a58512490eb6d
  dvc_pipeline_stages            features,split
  git_commit                     694fa803f80447ae330dfb1cbf5e9b9d8a9097d9-dirty

git 694fa803f804 / data.interim.features.parquet=d0605019, data.processed=5f68eff3, reports.split_summary.json=bf9ed4c8


A `-dirty` suffix on `git_commit` means uncommitted changes. Fine while
experimenting; CI commits first, so production runs always carry a clean sha.

## 2. Hyperparameters

From `params.yaml`, not from a notebook cell — so CI trains the same model you
just did, and a hyperparameter change shows up as a reviewable diff.

In [4]:
print(json.dumps(params["train"], indent=2))

{
  "xgboost_version": "1.7-1",
  "instance_type": "ml.m5.xlarge",
  "instance_count": 1,
  "max_run_seconds": 3600,
  "hyperparameters": {
    "num_round": 100,
    "max_depth": 3,
    "eta": 0.5,
    "alpha": 2.5,
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 3,
    "early_stopping_rounds": 10,
    "verbosity": 1
  }
}


## 3. Run the job

~4-5 minutes on `ml.m5.xlarge`. The call logs params, fits, then logs the
training metrics and the model artifact into one run.

In [5]:
from src.churn.training import train

result = train(settings=settings, params=params)

print("\nMLflow run :", result.run_id)
print("training job:", result.training_job_name)
print("model data  :", result.model_data)
print("metrics     :", json.dumps(result.metrics, indent=2))

/Users/ritabanchowdhury/.venvs/churn-mlops/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sagemaker.config INFO - Not applying SDK defaults from location: /usr/local/share/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/ritabanchowdhury/Library/Application Support/sagemaker/config.yaml


/Users/ritabanchowdhury/.venvs/churn-mlops/lib/python3.11/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
/Users/ritabanchowdhury/.venvs/churn-mlops/lib/python3.11/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See

2026-10-28 15:53:19 Starting - Starting the training job...
2026-10-28 15:53:34 Starting - Preparing the instances for training...
2026-10-28 15:53:57 Downloading - Downloading input data...
2026-10-28 15:54:28 Downloading - Downloading the training image...
2026-10-28 15:55:24 Training - Training image download completed. Training in progress.
2026-10-28 15:55:24 Uploading - Uploading generated training model./miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-10-28 15:55:18.792 ip-10-0-140-152.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-10-28 15:55:18.858 ip-10-0-140-152.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-10-28:15:

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
2026/10/28 11:56:21 WARNING mlflow.utils.environment: Failed to resolve installed pip vers

🏃 View run train-20261028-155311 at: https://us-east-1.experiments.sagemaker.aws/#/experiments/1/runs/0eb9b0c6f641470abae33de9b38e890a
🧪 View experiment at: https://us-east-1.experiments.sagemaker.aws/#/experiments/1

MLflow run : 0eb9b0c6f641470abae33de9b38e890a
training job: player-churn-xgb-2026-10-28-15-53-18-548
model data  : s3://sagemaker-us-east-1-063676201336/player-churn/dvc-mlflow/runs/20261028-155311/output/player-churn-xgb-2026-10-28-15-53-18-548/output/model.tar.gz
metrics     : {
  "validation_auc": 0.99372,
  "train_auc": 1.0
}


## 4. Inspect the run

Everything needed to reproduce this model, in one place.

In [6]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
run = MlflowClient().get_run(result.run_id)

print("params:")
for key, value in sorted(run.data.params.items()):
    print(f"  {key:28s} {value}")

print("\nmetrics:")
for key, value in sorted(run.data.metrics.items()):
    print(f"  {key:28s} {value}")

print("\nlineage tags:")
for key, value in sorted(run.data.tags.items()):
    if key.startswith("dvc") or key == "git_commit":
        print(f"  {key:28s} {value}")

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


params:
  hp.alpha                     2.5
  hp.colsample_bytree          0.8
  hp.early_stopping_rounds     10
  hp.eta                       0.5
  hp.eval_metric               auc
  hp.max_depth                 3
  hp.min_child_weight          3
  hp.num_round                 100
  hp.objective                 binary:logistic
  hp.subsample                 0.8
  hp.verbosity                 1
  instance_type                ml.m5.xlarge
  train_data_s3                s3://sagemaker-us-east-1-063676201336/player-churn/dvc-mlflow/runs/20261028-155311/input/train/train.csv
  training_job_name            player-churn-xgb-2026-10-28-15-53-18-548
  validation_data_s3           s3://sagemaker-us-east-1-063676201336/player-churn/dvc-mlflow/runs/20261028-155311/input/validation/validation.csv
  xgboost_version              1.7-1

metrics:
  train_auc                    1.0
  validation_auc               0.99372

lineage tags:
  dvc.data.interim.features.parquet d0605019cf55f50aaeee3c993fe9847a

In [7]:
# Save the handles for notebook 04 (and so CI can read the same file).
Path("reports").mkdir(exist_ok=True)
Path("reports/train_result.json").write_text(
    json.dumps(
        {
            "run_id": result.run_id,
            "training_job_name": result.training_job_name,
            "model_data": result.model_data,
            "metrics": result.metrics,
        },
        indent=2,
    ),
    encoding="utf-8",
)
print(Path("reports/train_result.json").read_text())

{
  "run_id": "0eb9b0c6f641470abae33de9b38e890a",
  "training_job_name": "player-churn-xgb-2026-10-28-15-53-18-548",
  "model_data": "s3://sagemaker-us-east-1-063676201336/player-churn/dvc-mlflow/runs/20261028-155311/output/player-churn-xgb-2026-10-28-15-53-18-548/output/model.tar.gz",
  "metrics": {
    "validation_auc": 0.99372,
    "train_auc": 1.0
  }
}


## 5. Reproducing this run later

```sh
git checkout <git_commit tag from the run>
dvc checkout            # restores the exact bytes
python -m churn.training
```

That is the whole argument for this stack. An MLflow metric with no dataset
pointer is a number you cannot defend.

In [8]:
# The CLI equivalent of this notebook, used verbatim by the train workflow.
print("python -m churn.training --run-name manual-$(date +%Y%m%d-%H%M%S)")

python -m churn.training --run-name manual-$(date +%Y%m%d-%H%M%S)


---
## Next

**`04-evaluate.ipynb`** -- score the held-out test split and apply the promotion gate.

### What changed vs. the original workshop
Old notebook 03 pasted an MLflow ARN into a cell, hardcoded `region = "us-east-1"`,
and wrote `model_s3_path` into a JSON file on S3 for the next notebook to read.
Here the ARN comes from `.env`, the region from `params.yaml`, and the handoff is
an MLflow run id. The run is also tagged with its data hashes, which the original
had no way to express.